In [1]:
import os
os.makedirs("output2", exist_ok=True)

In [2]:
import pandas as pd

# --- Load only needed columns ---
movies = pd.read_csv("data/movies.csv", usecols=["movieId", "title", "genres"])
links  = pd.read_csv("data/links.csv", usecols=["movieId", "imdbId", "tmdbId"])

print("Raw shapes:", movies.shape, links.shape)

# --- Clean movies table ---
movies_clean = movies[["movieId", "title", "genres"]].copy()

# optional: remove rows with missing essential values
movies_clean = movies_clean.dropna(subset=["movieId", "title"]).copy()

print("Movies cleaned shape:", movies_clean.shape)

# --- Clean links table ---
links_clean = links[["movieId", "imdbId", "tmdbId"]].copy()

# remove rows without imdbId
links_clean = links_clean.dropna(subset=["imdbId"]).copy()

# standardize imdbId into IMDb tconst format
links_clean["imdbId"] = links_clean["imdbId"].astype(int).astype(str)
links_clean["tconst"] = "tt" + links_clean["imdbId"].str.zfill(7)

print("Links cleaned shape:", links_clean.shape)

# --- Save cleaned CSVs separately ---
movies_clean.to_csv("output2/movies_clean.csv", index=False)
links_clean.to_csv("output2/links_clean.csv", index=False)

print("Saved: output2/movies_clean.csv")
print("Saved: output2/links_clean.csv")

Raw shapes: (87585, 3) (87585, 3)
Movies cleaned shape: (87585, 3)
Links cleaned shape: (87585, 4)
Saved: output2/movies_clean.csv
Saved: output2/links_clean.csv


In [3]:
import pandas as pd

BASICS_PATH  = "data/title.basics.tsv"
RATINGS_PATH = "data/title.ratings.tsv"

chunksize = 500_000  # adjust if your laptop memory is tight

# --- A) Filter title.basics.tsv to movies released in 2020-2025 ---
basics_parts = []

for chunk in pd.read_csv(BASICS_PATH, sep="\t", dtype=str, chunksize=chunksize):
    # Keep only movies
    chunk = chunk[chunk["titleType"] == "movie"].copy()

    # Convert startYear to numeric; IMDb uses '\N' for missing values
    chunk["startYear_num"] = pd.to_numeric(chunk["startYear"], errors="coerce")

    # Filter release years 2020-2025
    chunk = chunk[(chunk["startYear_num"] >= 2020) & (chunk["startYear_num"] <= 2025)]

    # Keep only needed columns
    basics_parts.append(chunk[["tconst", "primaryTitle", "startYear_num", "genres"]])

imdb_basics = pd.concat(basics_parts, ignore_index=True)
imdb_basics = imdb_basics.rename(columns={
    "primaryTitle": "imdb_title",
    "startYear_num": "start_year",
    "genres": "imdb_genres"
})

# Drop rows with missing key fields
imdb_basics = imdb_basics.dropna(subset=["tconst", "imdb_title", "start_year"])

print("IMDb basics filtered shape:", imdb_basics.shape)

# Create a set of relevant tconst to filter ratings efficiently
tconst_set = set(imdb_basics["tconst"].dropna().unique())
print("Unique tconst kept:", len(tconst_set))

# --- B) Filter title.ratings.tsv to only those tconst ---
ratings_parts = []

for chunk in pd.read_csv(RATINGS_PATH, sep="\t", dtype=str, chunksize=chunksize):
    chunk = chunk[chunk["tconst"].isin(tconst_set)].copy()
    ratings_parts.append(chunk[["tconst", "averageRating", "numVotes"]])

imdb_ratings = pd.concat(ratings_parts, ignore_index=True)

# Convert rating/votes to numeric
imdb_ratings["averageRating"] = pd.to_numeric(imdb_ratings["averageRating"], errors="coerce")
imdb_ratings["numVotes"] = pd.to_numeric(imdb_ratings["numVotes"], errors="coerce")

# Rename columns for clarity in Oracle
imdb_ratings = imdb_ratings.rename(columns={
    "averageRating": "imdb_avg_rating",
    "numVotes": "imdb_num_votes"
})

# Drop rows with missing key fields
imdb_ratings = imdb_ratings.dropna(subset=["tconst", "imdb_avg_rating", "imdb_num_votes"])

print("IMDb ratings filtered shape:", imdb_ratings.shape)

# --- C) Save cleaned IMDb CSVs separately ---
imdb_basics.to_csv("output2/imdb_basics_clean_2020_2025.csv", index=False)
imdb_ratings.to_csv("output2/imdb_ratings_clean_2020_2025.csv", index=False)

print("Saved: output2/imdb_basics_clean_2020_2025.csv")
print("Saved: output2/imdb_ratings_clean_2020_2025.csv")

IMDb basics filtered shape: (118611, 4)
Unique tconst kept: 118611
IMDb ratings filtered shape: (62373, 3)
Saved: output2/imdb_basics_clean_2020_2025.csv
Saved: output2/imdb_ratings_clean_2020_2025.csv


In [4]:
import pandas as pd

TMDB_PATH = "data/TMDB_movie_dataset_v11.csv"

# Load TMDB dataset
tmdb = pd.read_csv(TMDB_PATH, low_memory=False)

print("TMDB raw shape:", tmdb.shape)

# --- Convert release_date to datetime ---
tmdb["release_date"] = pd.to_datetime(tmdb["release_date"], errors="coerce")

# Extract year
tmdb["release_year"] = tmdb["release_date"].dt.year

# --- Filter movies released between 2020 and 2025 ---
tmdb = tmdb[
    (tmdb["release_year"] >= 2020) &
    (tmdb["release_year"] <= 2025)
].copy()

print("After year filter:", tmdb.shape)

# --- Clean imdb_id ---
# imdb_id looks like "tt1375666"
tmdb = tmdb.dropna(subset=["imdb_id"])

# --- Convert numeric fields ---
tmdb["vote_average"] = pd.to_numeric(tmdb["vote_average"], errors="coerce")
tmdb["vote_count"]   = pd.to_numeric(tmdb["vote_count"], errors="coerce")

# --- Keep only necessary columns ---
tmdb_clean = tmdb[
    [
        "id",
        "imdb_id",
        "title",
        "release_year",
        "genres",
        "vote_average",
        "vote_count"
    ]
].copy()

# Rename columns for clarity
tmdb_clean = tmdb_clean.rename(columns={
    "id": "tmdb_id",
    "title": "tmdb_title",
    "genres": "tmdb_genres",
    "vote_average": "tmdb_vote_average",
    "vote_count": "tmdb_vote_count"
})

print("TMDB clean final shape:", tmdb_clean.shape)

# --- Save cleaned CSV ---
tmdb_clean.to_csv("output2/tmdb_clean_2020_2025.csv", index=False)

print("Saved: output2/tmdb_clean_2020_2025.csv")

TMDB raw shape: (1380490, 24)
After year filter: (266110, 25)
TMDB clean final shape: (81343, 7)
Saved: output2/tmdb_clean_2020_2025.csv
